# Project: Wildfire Mapping

Goal: Build an html side with an interactive map where the user can see the recents wildfires. Provide the user with information of phisical size, duration, intensity, etc. of the wildfires. Use pop-ups and tooltips to make the map interactive and structured for the users. The map should be for public users which are interesting in wildfires.

Tasks:
1. Load the api
2. Extract the data which is used to locate the wildfire (VIIRS_SNPP_NRT)
3. Explore the data
4. Clean the data (if needed)
5. Check wildfires for different properties
6. Visulaization of the wildfires
7. Provide additional information about the wildfires. 
8. Make the map interactive

---

### Import which are used in this notebook
Make sure that all libraries are installed on the python envirnonment which is used to run this notebook

In [5]:
# import all libraries
import requests
import pandas as pd
import io
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from datetime import datetime
from shapely.geometry import MultiPoint
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import json
import numpy as np
import plotly.express as px
import os
from dotenv import load_dotenv

---
### Creating the API and it's key
After all libraries are imported, the next step is to get build the API url including its key.  
Create an key on: https://firms.modaps.eosdis.nasa.gov/api/map_key/  
Safe the map key in an .env file with the variable "FIRMS_API_KEY"


In [6]:
# Building the api url 

# import the api key fro the .env environment
load_dotenv()

#get the api key from the .env file
api_key = os.environ.get("FIRMS_API_KEY")
if not api_key:
    raise ValueError("FIRMS_API_KEY not set. Check your .env file.")

# define source
api_source = "VIIRS_SNPP_NRT" # or change ot to VIIRS_SNPP_SP?
# define area coordinates
api_area_coordinates = "world"
# day range. Days going back from today
api_day_range = 5
# build api url with api key
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{api_key}/{api_source}/{api_area_coordinates}/{api_day_range}"

After sucessfully creating the API url consisting the source, the location, the day-range and its key the api is loaded in

In [7]:
# load the data form the api
response = requests.get(api_url)

# check if api import was successfull
if response.status_code == 200:
    print("API request successfull")

    # get the data as a csv
    data_csv = response.text
    # create a dataframe
    data_df = pd.read_csv(io.StringIO(data_csv))

else:
    print(f"Request failed. Status code: {response.status_code}")

API request successfull


After loading succesfully the data from the api, in the next step the data gets referneced as a GeoDataFrame, named appropriatly, get cleaned and adjusted with a datetime format.  
Also some buffer around each fire are created. The goal of this buffer is to combine fires which occure on the same date in the buffer area. With this adjustment the fires which are nearby eachother will be referd to 1 fire and not several. This because it's likely that there are not two small fires but rather one big fire in this areas. This make the maps that are build later less crowded and therefore more easy to read and understand. There might be a loose in the visibility of fires in large areas, but we will take care of this in a later step

In [8]:
# converting the data data frame into a geo-data frame
wildfire_gdf_crs4326 = gpd.GeoDataFrame(data_df, geometry=gpd.points_from_xy(data_df["longitude"], data_df["latitude"]), crs=4326)

# convert the date column into a date datetime type
wildfire_gdf_crs4326["acq_date"] = pd.to_datetime(wildfire_gdf_crs4326["acq_date"], format="%Y-%m-%d")

# add a column with the date
wildfire_gdf_crs4326["acq_date_day"] = wildfire_gdf_crs4326["acq_date"].dt.date

# project to calculate in meters
wildfire_gdf_crs3857 = wildfire_gdf_crs4326.to_crs(epsg=3857)
# join nearby spatial points according to the same fire (most likly the same)
wildfire_gdf_crs3857["geometry_buffer"] = wildfire_gdf_crs3857.geometry.buffer(1501) # buffer of 1501 meters. resolution of the data is 350mx750m
# join fires within the buffer
joined_wildfires = gpd.sjoin(
    wildfire_gdf_crs3857[["acq_date", "geometry"]],
    wildfire_gdf_crs3857[["acq_date", "geometry_buffer"]].set_geometry("geometry_buffer"),
    how="left", # make sure to keep all points
    predicate="within"
)

joined_wildfires = joined_wildfires[joined_wildfires["acq_date_left"] == joined_wildfires["acq_date_right"]] # Keep only fires with the same registration date
# aggregate into clusters
wildfire_clusters = (
    joined_wildfires.groupby("index_right").agg(
        acq_date=("acq_date_left", "first"),
        count=("acq_date_left", "count"),
        geometry=("geometry", lambda geoms: MultiPoint(list(geoms)).centroid)
    )
    .reset_index(drop=True)
)

# Build the cleaned aggregated wildfire data
wildfire_clusterd = gpd.GeoDataFrame(wildfire_clusters, geometry="geometry", crs=3857).to_crs(epsg=4326)


In the next section we check the properties of the loaded fires from the api and form the clusterd fires. In this step one can check if the data import and cleaning happend as it should. 

In [9]:
# verify transformation to geo data frame
display(wildfire_gdf_crs4326.head(5))
display(wildfire_clusterd.head(5))
display(wildfire_clusterd.info)
display(wildfire_clusterd.dtypes)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,geometry,acq_date_day
0,19.40301,-155.28069,334.02,0.44,0.62,2026-05-14,24,N,VIIRS,l,2.0NRT,307.02,5.56,D,POINT (-155.28069 19.40301),2026-05-14
1,19.40339,-155.28026,335.12,0.44,0.62,2026-05-14,24,N,VIIRS,l,2.0NRT,307.07,5.79,D,POINT (-155.28026 19.40339),2026-05-14
2,19.40340,-155.27641,348.34,0.44,0.62,2026-05-14,24,N,VIIRS,n,2.0NRT,315.96,15.00,D,POINT (-155.27641 19.4034),2026-05-14
3,19.40379,-155.27216,350.93,0.44,0.62,2026-05-14,24,N,VIIRS,n,2.0NRT,318.12,15.00,D,POINT (-155.27216 19.40379),2026-05-14
4,19.40382,-155.27597,348.47,0.44,0.62,2026-05-14,24,N,VIIRS,n,2.0NRT,315.79,14.87,D,POINT (-155.27597 19.40382),2026-05-14


,acq_date,count,geometry
0,2026-05-14,24,POINT (-155.27905 19.40637)
1,2026-05-14,24,POINT (-155.27905 19.40637)
2,2026-05-14,22,POINT (-155.27814 19.40606)
3,2026-05-14,18,POINT (-155.27648 19.4054)
4,2026-05-14,22,POINT (-155.27814 19.40606)


<bound method DataFrame.info of          acq_date  count                     geometry
0      2026-05-14     24  POINT (-155.27905 19.40637)
1      2026-05-14     24  POINT (-155.27905 19.40637)
2      2026-05-14     22  POINT (-155.27814 19.40606)
3      2026-05-14     18   POINT (-155.27648 19.4054)
4      2026-05-14     22  POINT (-155.27814 19.40606)
...           ...    ...                          ...
129880 2026-05-18      5   POINT (-79.80816 43.26389)
129881 2026-05-18      2   POINT (-79.82428 43.27868)
129882 2026-05-18      2   POINT (-79.82428 43.27868)
129883 2026-05-18      1   POINT (-76.87791 43.28542)
129884 2026-05-18      1   POINT (-73.13916 46.04395)

[129885 rows x 3 columns]>

acq_date    datetime64[us]
count                int64
geometry          geometry
dtype: object

---
### Building a map showing the recent fire occurence on the world
Now all the data we need to build a first map with all the fires location are in the notebook.
In the next section an interactive map is produced. The map will show where the fire was recorded and how many fire detections were clusterd in this location. The number of Detections will give additional information about the spread of the fire.

In [10]:
# initalize Folium back ground map
background_map = folium.Map(
    location=[0, 0], # start zoom at latitude and longitude 0
    zoom_start=2, # shows the whole word at the start
    tiles="CartoDB DarkMatter", # Dark basmap
    control_scale=True # Add scalebar
)

cluster_fire = MarkerCluster(name="Recent Fires").add_to(background_map)
# build markers for the wildfires
for idx, row in wildfire_clusterd.iterrows():
    lat = row.geometry.y # extract latitude out of the geometry column 
    lon = row.geometry.x # eextract longitude out of the geometry column
    count = row["count"] # counting how many fires are aggregated

    # Formating the Tooltip. Date of the fire registation
    fire_start = row["acq_date"].date()
    tooltip = f"Date: {fire_start} | Detections: {count}"

    # Define Marker color deoending the count of fires
    if count == 1:
        color =  "orange"
    elif count <= 5:
        color = "red"
    else:
        color = "darkred"

    # create Marker of the fire locations
    folium.Marker(
        location=[lat, lon],
        tooltip=tooltip,
        icon=folium.Icon(color=color, icon="fire", prefix="fa")
    ).add_to(cluster_fire)

folium.LayerControl().add_to(background_map)
background_map.save("../outputs/map.html")

---
### Analyses of the Distribution of Fires
In addition map I also decided to give some Insighs about in which region most open fires are recorded. To start this analyses in a first step, data which provide data about the shape of each country must be loaded. Luckily, naturalearthdata.com provides data about each country, the borders and its location.  
Since the data form naturalearthdata.com is a huge dataset, the data is filtered just for data thats intersting for the preject. This are the Name of the Countries, the Continent of each country and its geometry. In addition, the area of each country is calculated, based on the geometry attribute. The area is later used to calculate the density of fires for each country.  
Also this data gets inspected to check if its loaded correctly.

In [11]:
# load Geo Data frame of the world countries
world_import = gpd.read_file("https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip").to_crs(epsg=4326) # load Countries from online source.

# inspect worl gdf
display(world_import.head(5))
print(world_import.columns.tolist())

# cleaning the world 
world = world_import[["NAME", "CONTINENT", "geometry"]] # just keep usefull attributes ot of the world gdf.
world["AREA_KM2"] = world.to_crs(epsg=3857).geometry.area / 1e6 # add and calculate new column with the area 
display(world)

,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,0,2,Indonesia,IDN,0,2,Sovereign country,1,Indonesia,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
1,Admin-0 country,0,3,Malaysia,MYS,0,2,Sovereign country,1,Malaysia,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((117.70361 4.16341, 117.69711 4..."
2,Admin-0 country,0,2,Chile,CHL,0,2,Sovereign country,1,Chile,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-69.51009 -17.50659, -69.50611..."
3,Admin-0 country,0,3,Bolivia,BOL,0,2,Sovereign country,1,Bolivia,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-69.51009 -17.50659, -69.51009 -17.5..."
4,Admin-0 country,0,2,Peru,PER,0,2,Sovereign country,1,Peru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-69.51009 -17.50659, -69.63832..."


['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF', 'GEOUNIT', 'GU_A3', 'SU_DIF', 'SUBUNIT', 'SU_A3', 'BRK_DIFF', 'NAME', 'NAME_LONG', 'BRK_A3', 'BRK_NAME', 'BRK_GROUP', 'ABBREV', 'POSTAL', 'FORMAL_EN', 'FORMAL_FR', 'NAME_CIAWF', 'NOTE_ADM0', 'NOTE_BRK', 'NAME_SORT', 'NAME_ALT', 'MAPCOLOR7', 'MAPCOLOR8', 'MAPCOLOR9', 'MAPCOLOR13', 'POP_EST', 'POP_RANK', 'POP_YEAR', 'GDP_MD', 'GDP_YEAR', 'ECONOMY', 'INCOME_GRP', 'FIPS_10', 'ISO_A2', 'ISO_A2_EH', 'ISO_A3', 'ISO_A3_EH', 'ISO_N3', 'ISO_N3_EH', 'UN_A3', 'WB_A2', 'WB_A3', 'WOE_ID', 'WOE_ID_EH', 'WOE_NOTE', 'ADM0_ISO', 'ADM0_DIFF', 'ADM0_TLC', 'ADM0_A3_US', 'ADM0_A3_FR', 'ADM0_A3_RU', 'ADM0_A3_ES', 'ADM0_A3_CN', 'ADM0_A3_TW', 'ADM0_A3_IN', 'ADM0_A3_NP', 'ADM0_A3_PK', 'ADM0_A3_DE', 'ADM0_A3_GB', 'ADM0_A3_BR', 'ADM0_A3_IL', 'ADM0_A3_PS', 'ADM0_A3_SA', 'ADM0_A3_EG', 'ADM0_A3_MA', 'ADM0_A3_PT', 'ADM0_A3_AR', 'ADM0_A3_JP', 'ADM0_A3_KO', 'ADM0_A3_VN', 'ADM0_A3_TR', 'AD

,NAME,CONTINENT,geometry,AREA_KM2
0,Indonesia,Asia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...",1.901567e+06
1,Malaysia,Asia,"MULTIPOLYGON (((117.70361 4.16341, 117.69711 4...",3.317439e+05
2,Chile,South America,"MULTIPOLYGON (((-69.51009 -17.50659, -69.50611...",1.255936e+06
3,Bolivia,South America,"POLYGON ((-69.51009 -17.50659, -69.51009 -17.5...",1.194826e+06
4,Peru,South America,"MULTIPOLYGON (((-69.51009 -17.50659, -69.63832...",1.339975e+06
...,...,...,...,...
253,Macao,Asia,"MULTIPOLYGON (((113.5586 22.16303, 113.56943 2...",3.523246e+01
254,Ashmore and Cartier Is.,Oceania,"POLYGON ((123.59702 -12.42832, 123.59775 -12.4...",2.843784e+00
255,Bajo Nuevo Bank,North America,"POLYGON ((-79.98929 15.79495, -79.98782 15.796...",3.895604e-02
256,Serranilla Bank,North America,"POLYGON ((-78.63707 15.86209, -78.64041 15.864...",1.144447e-01


After the succesfull import of the countries poroperties, to each fire a the country of its occurence gets assigned. In this step some fires might get lost. This is due to some fires occure/are recorded in on non territorials areas such like on the ocean or in Arctica. 
  
The get the numbers of fires per country the dataset with the fires and its corresponding name must be grouped by the country. To capture all countries (and some accepted territories), the countries with zero fire records must be added with zero recordings. The reason this is that in a later step on the map this counties without any fire recordings also can be impplemented.

In [12]:
# calculate fires in the diffrent countries
# here the wildfire cluster is used, so nearby fires are counted as one fire and not as several

wildfire_countries = gpd.sjoin(wildfire_clusterd, world, 
                                how="left", predicate="within") # assign to each fire in which country it is located

# Check for fires not located inside a country for example this are burning ships, fires in artica or antarctica
unmatched = wildfire_countries[wildfire_countries["NAME"].isna()]
print(f"Unmatched fires: {len(unmatched)}")
display(wildfire_countries.head(5))

# count the fires for each country
fire_per_country = world[["NAME", "CONTINENT", "geometry", "AREA_KM2"]].copy() # copy world to build fire per country
# merge fire counts in — countries with no fires get NaN
fire_per_country = fire_per_country.merge(
    wildfire_countries.groupby("NAME").size().reset_index(name="fire_count"),
    on="NAME",
    how="left"  # keep all countries from world
)

fire_per_country["fire_count"] = fire_per_country["fire_count"].fillna(0).astype(int) # fill the NaN values of the countries without fires with 0

fire_per_country = gpd.GeoDataFrame(fire_per_country, geometry="geometry", crs=4326) # 

display(fire_per_country)

Unmatched fires: 1549


,acq_date,count,geometry,index_right,NAME,CONTINENT,AREA_KM2
0,2026-05-14,24,POINT (-155.27905 19.40637),154.0,United States of America,North America,2.172158e+07
1,2026-05-14,24,POINT (-155.27905 19.40637),154.0,United States of America,North America,2.172158e+07
2,2026-05-14,22,POINT (-155.27814 19.40606),154.0,United States of America,North America,2.172158e+07
3,2026-05-14,18,POINT (-155.27648 19.4054),154.0,United States of America,North America,2.172158e+07
4,2026-05-14,22,POINT (-155.27814 19.40606),154.0,United States of America,North America,2.172158e+07


,NAME,CONTINENT,geometry,AREA_KM2,fire_count
0,Indonesia,Asia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...",1.901567e+06,306
1,Malaysia,Asia,"MULTIPOLYGON (((117.70361 4.16341, 117.69711 4...",3.317439e+05,30
2,Chile,South America,"MULTIPOLYGON (((-69.51009 -17.50659, -69.50611...",1.255936e+06,632
3,Bolivia,South America,"POLYGON ((-69.51009 -17.50659, -69.51009 -17.5...",1.194826e+06,1051
4,Peru,South America,"MULTIPOLYGON (((-69.51009 -17.50659, -69.63832...",1.339975e+06,320
...,...,...,...,...,...
253,Macao,Asia,"MULTIPOLYGON (((113.5586 22.16303, 113.56943 2...",3.523246e+01,0
254,Ashmore and Cartier Is.,Oceania,"POLYGON ((123.59702 -12.42832, 123.59775 -12.4...",2.843784e+00,0
255,Bajo Nuevo Bank,North America,"POLYGON ((-79.98929 15.79495, -79.98782 15.796...",3.895604e-02,0
256,Serranilla Bank,North America,"POLYGON ((-78.63707 15.86209, -78.64041 15.864...",1.144447e-01,0


---
### Mapping Fire per Country
In this section the map of the fires per country gets build. Here the area of the countries is not considered. This means that larger countries tend to have more fires, because there is more potential areas for fire. In the section after this one the countries size will be considered. 

In [13]:
# make a map displaying the nubers of fire per each country
geojson = json.loads(fire_per_country.to_json())


max_fires = fire_per_country["fire_count"].max()
# custom colorscale: grey for 0, then OrRd for fires
colorscale = [
    [0, "lightgrey"],
    [0.0001, "lightyellow"],
    [0.5, "orange"],
    [1, "darkred"]
]

fig = go.Figure(go.Choropleth(
    geojson=geojson,
    locations=fire_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_per_country["fire_count"],
    colorscale=colorscale,
    colorbar_title="Fires recorded",
    customdata=fire_per_country[["NAME", "fire_count"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Fires recorded: %{customdata[1]}<extra></extra>",
))

fig.update_layout(
    title=dict(text="Fires per Country", x=0.5, xanchor="center", font=dict(size=24)),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
    )
)

fig.write_html("../outputs/fire_per_country.html")

### Mapping the Fire considering Country Size
As already mentioned, just the fire per country are not really informative. Large countries tend to recored many fires where in smaller countries less fire tend to be recorded. Thats why in this section the density of the fire per country gets introduced. This give some good insight in which country the amount of covering a lot of area. However, since we still work with the clusterd fire, this map will not provide any fix information of the area of fire records. One big fire is counted as with the same value as a small fire.  
  
Because the distrubition of in the fire denisty is right-skewed mapping with linear scale does show many countries with a low density and just highlights the few country with a high density. Therefore, also the density with a logarithmic scale is calulated. Still the destribution is right-skewed but much better distributed than with the linear scale

In [14]:
# make a map with density of fires per country
fire_density_per_country = fire_per_country.copy() 
fire_density_per_country["density_100km2"] = fire_density_per_country["fire_count"] / (fire_density_per_country["AREA_KM2"]) *10000 # add clolumn density per 100km2

# add log column, log(0) is undefined so use log(x+1)
fire_density_per_country["density_log"] = np.log1p(fire_density_per_country["density_100km2"])

# inspect the distributen of the density
fire_density_per_country.describe()

,AREA_KM2,fire_count,density_100km2,density_log
count,2.580000e+02,258.000000,258.000000,258.000000
mean,3.423154e+07,497.426357,8.068444,1.044585
std,5.295922e+08,2001.211354,22.174830,1.279563
min,2.204709e-02,0.000000,0.000000,0.000000
25%,9.801617e+02,0.000000,0.000000,0.000000
50%,8.692411e+04,7.000000,0.685503,0.521913
75%,5.411109e+05,162.750000,4.850489,1.766523
max,8.507102e+09,21073.000000,189.381601,5.249030


In the next section the Density map showing fires per 100km2 is build. This map will show the fires density in a linear scale. As mentioned before, a lot of counties have a low density and therfore will be diffucult to distingish.

In [15]:
# render map linar scale
geojson_density = json.loads(fire_density_per_country.to_json())

# color scale
colorscale = [
    [0, "lightgrey"],
    [0.0001, "lightyellow"],
    [0.5, "orange"],
    [1, "darkred"]
]

fig = go.Figure(go.Choropleth(
    geojson=geojson_density,
    locations=fire_density_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_density_per_country["density_100km2"],
    colorscale=colorscale,
    colorbar_title="Fires per 100km²",
    customdata=fire_density_per_country[["NAME", "fire_count", "density_100km2"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Fires recorded: %{customdata[1]}<br>Fires per 100km²: %{customdata[2]:.2f}<extra></extra>",
))

fig.update_layout(
    title=dict(text="Fire Density per 100km² by Country", x=0.5, xanchor="center", font=dict(size=24)),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
    )
)

fig.write_html("../outputs/fire_density_per_country_linear.html")

Since the density fire map with the linear scale is not very informative. The next section will be about the logarithmic scale. One disadvantege of this map is, that just the colorscale is not even over its lenght. In the high density part just a small colorchange means a large increase in fire density, while in the lower scale a small colorchange is not such a big increase in fire density.

In [16]:
# render map logarithmic scale
geojson_density = json.loads(fire_density_per_country.to_json())

# color scale
colorscale = [
    [0, "lightgrey"],
    [0.0001, "lightyellow"],
    [0.5, "orange"],
    [1, "darkred"]
]

fig = go.Figure(go.Choropleth(
    geojson=geojson_density,
    locations=fire_density_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_density_per_country["density_log"],
    colorscale=colorscale,
    colorbar_title="Fires per 100km²",
    customdata=fire_density_per_country[["NAME", "fire_count", "density_100km2"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Fires recorded: %{customdata[1]}<br>Fires per 100km²: %{customdata[2]:.2f}<extra></extra>",
))

fig.update_layout(
    title=dict(text="Fire Density per 100km² by Country", x=0.5, xanchor="center", font=dict(size=24)),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
    )
)

fig.write_html("../outputs/fire_density_per_country_logarithmic.html")

---
### Fire Heatmap
Till now we just looked at the different fires and the countries within the fire was recorded. As we all know a fire migh not just stop at a countries border (if the border isn't a big river, lake or high-mountain ridge). For a whole overview the see which regions are affected by fire, the heat map, which is build in the next section is very useful. The heatmap will highlight region where many fires were recorded. For this case the cleaned loaded api data is used and not the clusterd one. This helps to really highlight the fires even zoomed in on a smaller scale. 

In [17]:
# make a heatmap of the fires
fig = px.density_map(
    wildfire_gdf_crs4326,
    lat=wildfire_gdf_crs4326.geometry.y,
    lon=wildfire_gdf_crs4326.geometry.x,
    radius=5,
    zoom=1,
    map_style="carto-darkmatter",
    title="Wildfire Heatmap",
    color_continuous_scale="Hot",
    hover_data={"acq_date_day": True}
)

fig.update_traces(hovertemplate="Date: %{customdata[0]}<extra></extra>")

fig.update_layout(
    title=dict(text="Global Wildfire Heatmap", x=0.5, xanchor="center", font=dict(size=24)),
    coloraxis_showscale=False,
)

fig.write_html("../outputs/fire_heatmap.html")

---
### From visual to numbers
The maps before just provide some visual information. It is really diffucult to get the real number out of the maps. Therefore, in this a dataset is created taking the numbers for each country.

In [20]:
# Sorting the fire per country and print it
fire_per_country.sort_values("fire_count", ascending=False)[["NAME", "fire_count"]].to_csv("../outputs/fire_per_country.csv", index=False)

# sorting the country with most fire per density
fire_density_per_country.sort_values("density_100km2", ascending=False)[["NAME", "density_100km2", "AREA_KM2"]].to_csv("../outputs/fire_density_per_country.csv", index=False)